# 04c — PhoBERT-base-v2 fine-tuning (Week 4 Phase 2)

Final phase of Week 4: fine-tune `vinai/phobert-base-v2` on cleaned ViHSD
and report a 3-way comparison against the Week-3 LR champion and the
Week-4 Phase 1 BiLSTM v2.

**Config from `04a` E2** (validated on this 6 GB RTX 3060):

* `batch_size = 16`, `max_len = 128`, `fp16 = True`, `grad_accum = 1`
* Peak VRAM @ E2: 2.81 GB (with safety margin to 5.0 GB ceiling)
* If you actually OOM mid-training: drop to `batch_size=8` + `grad_accum=2`

**Training stack:** HuggingFace `Trainer` with a custom `WeightedTrainer`
subclass that applies the same `[0.40, 4.99, 3.14]` class weights as the
BiLSTM pipeline (computed in `04a` C4 via `sklearn.compute_class_weight`).

**Filter rule:** same `min_length=3` as the BiLSTM pipeline, applied to
training only. Dev and test stay unfiltered for fair eval.

**Outputs:**

* `models/dl/phobert_best/`                          — final HF-style folder (model + tokenizer)
* `models/dl/phobert_checkpoints/`                   — Trainer checkpoints (auto-cleaned to last 2)
* `results/phobert_training_log.json`                — per-epoch metrics
* `results/phobert_dev_predictions.csv`              — per-sample predictions on dev
* `results/figures/18_phobert_dev_cm.png`            — confusion matrix on DEV
* `results/figures/19_phobert_training_curves.png`   — train/eval loss + per-class F1
* `results/figures/20_three_way_comparison.png`      — LR vs BiLSTM v2 vs PhoBERT
* `report/week4_phase2_summary.md`                   — auto-generated decision log


In [ ]:
# ── Setup: autoreload, paths, seed ──────────────────────────────────────
%load_ext autoreload
%autoreload 2

import sys
for _k in [k for k in sys.modules if k == "src" or k.startswith("src.")]:
    del sys.modules[_k]

import os, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed as hf_set_seed,
)

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, classification_report,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path = [str(ROOT)] + [p for p in sys.path if p != str(ROOT)]

from configs.config import PATHS, COLUMNS, LABEL_MAP, PHOBERT_CONFIG
from src.utils import set_seed, get_device

RANDOM_STATE = 42
set_seed(RANDOM_STATE)
hf_set_seed(RANDOM_STATE)
device = get_device()

DL_DIR        = ROOT / "models" / "dl"
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR   = ROOT / "results"
FIG_DIR       = RESULTS_DIR / "figures"
REPORT_DIR    = ROOT / "report"
LOG_DIR       = ROOT / "logs" / "phobert"
PHOBERT_CKPT_DIR = DL_DIR / "phobert_checkpoints"
PHOBERT_BEST_DIR = DL_DIR / "phobert_best"
for d in (DL_DIR, RESULTS_DIR, FIG_DIR, REPORT_DIR, LOG_DIR, PHOBERT_CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

TEXT, LABEL = COLUMNS["text"], COLUMNS["label"]

PHOBERT_NAME = PHOBERT_CONFIG["pretrained_model"]
MAX_LEN      = PHOBERT_CONFIG["max_len"]       # 128 from 04a E2
BATCH_SIZE   = PHOBERT_CONFIG["batch_size"]    # 16
EVAL_BATCH   = 32
MIN_LENGTH   = 3                                # match BiLSTM filter
NUM_WORKERS  = 0                                # Jupyter+Windows: keep 0 to avoid hangs; bump to 2 elsewhere

print(f"Device       : {device}")
print(f"Model        : {PHOBERT_NAME}")
print(f"max_len      : {MAX_LEN}   batch_size: {BATCH_SIZE}   fp16: True")
print(f"min_length   : {MIN_LENGTH}  (train-only filter)")
print(f"seed         : {RANDOM_STATE}")


## A. Load PhoBERT model + tokenizer + verify

`use_fast=False` for PhoBERT — the fast tokenizer doesn't exist for this
model. Memory after load tells us how much VRAM is left for activations
and gradients.

In [ ]:
print(f"Loading tokenizer: {PHOBERT_NAME}")
tokenizer = AutoTokenizer.from_pretrained(PHOBERT_NAME, use_fast=False)

print(f"Loading model: {PHOBERT_NAME} ({3} labels)")
t0 = time.perf_counter()
model = AutoModelForSequenceClassification.from_pretrained(
    PHOBERT_NAME,
    num_labels=3,
    id2label={0: "CLEAN", 1: "OFFENSIVE", 2: "HATE"},
    label2id={"CLEAN": 0, "OFFENSIVE": 1, "HATE": 2},
).to(device)
print(f"  loaded in {time.perf_counter()-t0:.1f}s")

# Param count.
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  total params     : {total_params:>12,d}")
print(f"  trainable params : {trainable_params:>12,d}")

# Memory snapshot.
if device.type == "cuda":
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU memory       : {alloc:.2f} GB allocated / {reserved:.2f} GB reserved / {total_vram:.2f} GB total")

# Quick inference smoke test (one sample).
sample_text = "Hôm_nay trời rất đẹp ."
enc = tokenizer(sample_text, return_tensors="pt", padding="max_length",
                max_length=MAX_LEN, truncation=True).to(device)
model.eval()
with torch.no_grad():
    out = model(**enc)
probs = torch.softmax(out.logits, dim=-1).cpu().numpy()[0]
print(f"\nInference smoke test on {sample_text!r}:")
for i, p in enumerate(probs):
    print(f"  P({LABEL_MAP[i]:<10s}) = {p:.4f}")


In [ ]:
# Verify text is word-segmented (underthesea joins multi-syllable words with '_').
# PhoBERT-v2 was trained on segmented text; using raw text would be a silent
# accuracy hit.
train_df_check = pd.read_csv(PROCESSED_DIR / "train_cleaned.csv")
samples = train_df_check["cleaned"].dropna().sample(5, random_state=RANDOM_STATE).tolist()

print("Segmentation check (looking for '_'):")
has_underscore = []
for s in samples:
    has_underscore.append("_" in str(s))
    print(f"  [_:{('_' in str(s))!s:<5s}] {str(s)[:80]!r}")

if all(not u for u in has_underscore):
    print("\n⚠ NO underscores in any sample — text may NOT be word-segmented.")
    print("  PhoBERT-v2 was trained on underthesea-segmented text; re-run 02_preprocessing.ipynb.")
elif sum(has_underscore) >= 2:
    print(f"\n✓ Segmented text detected ({sum(has_underscore)}/{len(samples)} samples contain '_').")
else:
    print(f"\n? Only {sum(has_underscore)}/{len(samples)} samples contain '_' — may be a short-text sample.")


## B. Data preparation

Same `min_length=3` filter as `04b` (BiLSTM v2), applied via word-level
length so the *composition* of the training set matches across BiLSTM and
PhoBERT — keeps the comparison fair. The filter wrapper accepts an
explicit `length_tokenizer=vocab` argument for that purpose.

In [ ]:
from src.dataset_dl import Vocab, ViHSDDataset, FilteredViHSDDataset, collate_fn_phobert

train_df = pd.read_csv(PROCESSED_DIR / "train_cleaned.csv")
dev_df   = pd.read_csv(PROCESSED_DIR / "dev_cleaned.csv")
test_df  = pd.read_csv(PROCESSED_DIR / "test_cleaned.csv")

# Vocab is only used for the length-based filter (matches BiLSTM filter).
vocab = Vocab.load(DL_DIR / "vocab.pkl")

# Class weights from 04a C4 (sklearn 'balanced' → [0.40, 4.99, 3.14]).
class_weights = torch.load(DL_DIR / "class_weights.pt", weights_only=True)
print(f"class weights: {[f'{LABEL_MAP[i]}={w:.3f}' for i, w in enumerate(class_weights.tolist())]}\n")

train_ds_raw = ViHSDDataset(
    texts     = train_df["cleaned"].fillna("").tolist(),
    labels    = train_df[LABEL].tolist(),
    tokenizer = tokenizer,                              # HF tokenizer for actual inputs
    max_len   = MAX_LEN,
    mode      = "phobert",
)
train_ds = FilteredViHSDDataset(
    base             = train_ds_raw,
    min_length       = MIN_LENGTH,
    length_tokenizer = vocab,                           # word-level length, same as BiLSTM
    verbose          = True,
)

dev_ds = ViHSDDataset(
    texts     = dev_df["cleaned"].fillna("").tolist(),
    labels    = dev_df[LABEL].tolist(),
    tokenizer = tokenizer,
    max_len   = MAX_LEN,
    mode      = "phobert",
)
test_ds = ViHSDDataset(
    texts     = test_df["cleaned"].fillna("").tolist(),
    labels    = test_df[LABEL].tolist(),
    tokenizer = tokenizer,
    max_len   = MAX_LEN,
    mode      = "phobert",
)

print(f"\nTrain (filtered): {len(train_ds):,}")
print(f"Dev   (unfiltered): {len(dev_ds):,}")
print(f"Test  (unfiltered, NOT used in 04c): {len(test_ds):,}")


In [ ]:
# Verify 1 sample comes through correctly.
sample = train_ds[0]
print(f"sample[0] keys     : {list(sample.keys())}")
print(f"  input_ids shape  : {tuple(sample['input_ids'].shape)}  dtype: {sample['input_ids'].dtype}")
print(f"  attention_mask   : {tuple(sample['attention_mask'].shape)}  sum (real tokens): {int(sample['attention_mask'].sum())}")
print(f"  label            : {int(sample['label'])}  ({LABEL_MAP[int(sample['label'])]})")
print()
# Decode tokens (drop padding for clarity).
real_ids = sample["input_ids"][sample["attention_mask"].bool()].tolist()
tokens   = tokenizer.convert_ids_to_tokens(real_ids)
print(f"  tokens ({len(tokens)} real):  {tokens[:24]}{'…' if len(tokens) > 24 else ''}")
print(f"  decoded back     : {tokenizer.decode(real_ids, skip_special_tokens=True)!r}")


## C. Training with HuggingFace `Trainer`

`WeightedTrainer` is a one-line subclass that injects our pre-computed
class weights into `nn.CrossEntropyLoss`. All other behaviour (lr scheduler,
fp16, mixed-precision, gradient accumulation, checkpointing, early stopping)
comes from the stock `Trainer`.

In [ ]:
# Compute-metrics function called by Trainer at end of each eval epoch.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    per_f = f1_score(labels, preds, average=None, labels=[0, 1, 2], zero_division=0)
    return {
        "f1_macro":        float(f1_score(labels, preds, average="macro", zero_division=0)),
        "f1_clean":        float(per_f[0]),
        "f1_offensive":    float(per_f[1]),
        "f1_hate":         float(per_f[2]),
        "accuracy":        float(accuracy_score(labels, preds)),
        "precision_macro": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall_macro":    float(recall_score(labels, preds, average="macro", zero_division=0)),
    }


In [ ]:
# Trainer subclass that applies pre-computed class weights to CE loss.
# The **kwargs catches `num_items_in_batch` added in newer transformers.
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


In [ ]:
training_args = TrainingArguments(
    output_dir = str(PHOBERT_CKPT_DIR),

    # Training schedule
    num_train_epochs              = 5,
    per_device_train_batch_size   = BATCH_SIZE,
    per_device_eval_batch_size    = EVAL_BATCH,
    gradient_accumulation_steps   = 1,

    # Optimizer
    learning_rate                 = PHOBERT_CONFIG["learning_rate"],   # 2e-5
    warmup_ratio                  = PHOBERT_CONFIG.get("warmup_ratio", 0.1),
    weight_decay                  = PHOBERT_CONFIG.get("weight_decay", 0.01),
    max_grad_norm                 = 1.0,
    lr_scheduler_type             = "linear",

    # Eval & save (epoch-level so log_history is easy to parse later)
    eval_strategy                 = "epoch",
    save_strategy                 = "epoch",
    load_best_model_at_end        = True,
    metric_for_best_model         = "f1_macro",
    greater_is_better             = True,
    save_total_limit              = 2,

    # Logging
    logging_dir                   = str(LOG_DIR),
    logging_strategy              = "steps",
    logging_steps                 = 100,

    # Performance / 6 GB VRAM
    fp16                          = True,
    dataloader_num_workers        = NUM_WORKERS,
    dataloader_pin_memory         = True,

    # Reproducibility
    seed                          = RANDOM_STATE,
    data_seed                     = RANDOM_STATE,

    # Suppress 3rd-party logging
    report_to                     = "none",
    push_to_hub                   = False,
)

print(f"Output dir       : {training_args.output_dir}")
print(f"Epochs           : {training_args.num_train_epochs}")
print(f"Batch size       : {training_args.per_device_train_batch_size}  "
      f"(eff. {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps})")
print(f"lr / warmup / wd : {training_args.learning_rate} / {training_args.warmup_ratio} / {training_args.weight_decay}")
print(f"fp16 / num_work  : {training_args.fp16} / {training_args.dataloader_num_workers}")


In [ ]:
# Custom collator — same as src.dataset_dl.collate_fn_phobert: renames our
# 'label' key to HF's expected 'labels'.
def phobert_collate_fn(batch):
    return {
        "input_ids":      torch.stack([b["input_ids"]      for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels":         torch.stack([b["label"]          for b in batch]),
    }

trainer = WeightedTrainer(
    class_weights   = class_weights,
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = dev_ds,
    compute_metrics = compute_metrics,
    data_collator   = phobert_collate_fn,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Trainer initialised.")
print(f"  train samples : {len(train_ds):,}")
print(f"  dev samples   : {len(dev_ds):,}")
print(f"  steps/epoch   : ~{len(train_ds) // BATCH_SIZE:,}")
print(f"  total steps   : ~{(len(train_ds) // BATCH_SIZE) * int(training_args.num_train_epochs):,}")


In [ ]:
# ── Train ─────────────────────────────────────────────────────────────
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

print(f"Starting fine-tuning of {PHOBERT_NAME} ...")
print(f"Watch nvidia-smi in another terminal if you want to track VRAM live.")
print()

t0 = time.perf_counter()
train_result = trainer.train()
total_train_time = time.perf_counter() - t0

peak_vram = (torch.cuda.max_memory_allocated() / 1e9) if device.type == "cuda" else 0.0

print(f"\n✓ Training complete in {total_train_time:.1f}s ({total_train_time/60:.1f} min)")
print(f"  best metric (f1_macro) : {trainer.state.best_metric}")
print(f"  best model dir         : {trainer.state.best_model_checkpoint}")
print(f"  peak VRAM              : {peak_vram:.2f} GB")
print(f"  train_runtime          : {train_result.metrics.get('train_runtime', 0):.1f}s")
print(f"  train_steps_per_second : {train_result.metrics.get('train_steps_per_second', 0):.2f}")

# Save final (best) model + tokenizer to a stable location.
trainer.save_model(str(PHOBERT_BEST_DIR))
tokenizer.save_pretrained(str(PHOBERT_BEST_DIR))
print(f"\n✓ Final model + tokenizer saved → {PHOBERT_BEST_DIR}")

# Persist the full log history alongside the checkpoint.
with open(RESULTS_DIR / "phobert_training_log.json", "w", encoding="utf-8") as f:
    json.dump({
        "log_history":      trainer.state.log_history,
        "best_metric":      trainer.state.best_metric,
        "best_checkpoint":  trainer.state.best_model_checkpoint,
        "total_train_time": float(total_train_time),
        "peak_vram_gb":     float(peak_vram),
        "param_info": {
            "trainable": trainable_params, "total": total_params,
        },
        "config": {
            "model_name":      PHOBERT_NAME,
            "max_len":         MAX_LEN,
            "batch_size":      BATCH_SIZE,
            "fp16":            True,
            "learning_rate":   training_args.learning_rate,
            "weight_decay":    training_args.weight_decay,
            "warmup_ratio":    training_args.warmup_ratio,
            "num_train_epochs": int(training_args.num_train_epochs),
            "min_length":      MIN_LENGTH,
        },
    }, f, indent=2)
print(f"✓ training log → {RESULTS_DIR / 'phobert_training_log.json'}")


## D. Evaluate the best checkpoint on DEV

`load_best_model_at_end=True` already swapped the in-memory model to the
best epoch, but we reload from `phobert_best/` to keep eval state cleanly
separated from training state (and to prove the saved artefact actually
works for downstream).

In [ ]:
from src.evaluate import evaluate_model, plot_confusion_matrix, save_predictions

# Reload from the saved 'best' folder (proves the artefact is usable).
print(f"Reloading model from {PHOBERT_BEST_DIR} for clean eval ...")
eval_model = AutoModelForSequenceClassification.from_pretrained(str(PHOBERT_BEST_DIR)).to(device)
eval_model.eval()

dev_loader = DataLoader(
    dev_ds, batch_size=EVAL_BATCH, shuffle=False,
    collate_fn=phobert_collate_fn,
    num_workers=NUM_WORKERS, pin_memory=True,
)

all_preds, all_labels, all_probs = [], [], []
t0 = time.perf_counter()
with torch.no_grad():
    for batch in tqdm(dev_loader, desc="eval-dev"):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attn      = batch["attention_mask"].to(device, non_blocking=True)
        labels    = batch["labels"]
        # fp16 inference for parity with training, and ~2x faster.
        with torch.cuda.amp.autocast(dtype=torch.float16):
            out = eval_model(input_ids=input_ids, attention_mask=attn)
        logits = out.logits.float()
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        preds  = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
        all_probs.extend(probs.tolist())

dev_infer_time = time.perf_counter() - t0
y_dev_true = np.asarray(all_labels)
y_dev_pred = np.asarray(all_preds)
y_dev_proba = np.asarray(all_probs)
print(f"\nDEV inference: {dev_infer_time:.2f}s on {len(y_dev_true):,} samples "
      f"({dev_infer_time/len(y_dev_true)*1000:.2f} ms/sample)")


In [ ]:
phobert_metrics = evaluate_model(
    y_true         = y_dev_true,
    y_pred         = y_dev_pred,
    model_name     = "PhoBERT-base-v2 (fine-tuned)",
    train_time     = float(total_train_time),
    inference_time = float(dev_infer_time),
)

print(f"PhoBERT — DEV evaluation (best epoch by f1_macro)")
print(f"  accuracy : {phobert_metrics['accuracy']:.4f}")
print(f"  f1_macro : {phobert_metrics['f1_macro']:.4f}")
print(f"  f1 clean / off / hate : "
      f"{phobert_metrics['f1_clean']:.4f} / "
      f"{phobert_metrics['f1_offensive']:.4f} / "
      f"{phobert_metrics['f1_hate']:.4f}")
print()
print(phobert_metrics["classification_report"])

cm_path = FIG_DIR / "18_phobert_dev_cm.png"
plot_confusion_matrix(
    y_true     = y_dev_true,
    y_pred     = y_dev_pred,
    model_name = "PhoBERT-base-v2 (fine-tuned) — DEV",
    save_path  = cm_path,
    normalize  = True,
)
plt.show()
print(f"✓ saved → {cm_path}")

pred_csv = RESULTS_DIR / "phobert_dev_predictions.csv"
save_predictions(
    texts     = dev_df["cleaned"].fillna("").tolist(),
    y_true    = y_dev_true,
    y_pred    = y_dev_pred,
    y_proba   = y_dev_proba,
    save_path = pred_csv,
)
print(f"✓ saved → {pred_csv}")


In [ ]:
# Parse Trainer.state.log_history into per-epoch metrics.
log_hist = trainer.state.log_history
train_rows = [r for r in log_hist if "loss" in r and "eval_loss" not in r]
eval_rows  = [r for r in log_hist if "eval_loss" in r]

train_df_log = pd.DataFrame(train_rows)
eval_df_log  = pd.DataFrame(eval_rows)

print(f"log_history: {len(log_hist)} entries")
print(f"  train entries: {len(train_df_log)}  | eval entries: {len(eval_df_log)}")

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)

# Train loss curve (vs step) + eval loss (vs step at epoch boundaries).
ax = axes[0, 0]
if not train_df_log.empty:
    ax.plot(train_df_log["step"], train_df_log["loss"], "-", color="#4C78A8", alpha=0.6, label="train (step)")
if not eval_df_log.empty:
    ax.plot(eval_df_log["step"], eval_df_log["eval_loss"], "o-", color="#E45756", label="eval (epoch)")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_title("Loss"); ax.legend(); ax.grid(alpha=0.3)

# Eval F1 macro vs epoch.
ax = axes[0, 1]
if not eval_df_log.empty:
    ax.plot(eval_df_log["epoch"], eval_df_log["eval_f1_macro"], "o-", color="#54A24B", linewidth=2)
    best_eval = eval_df_log.loc[eval_df_log["eval_f1_macro"].idxmax()]
    ax.axvline(best_eval["epoch"], color="#54A24B", linestyle=":", alpha=0.6,
               label=f"best @ ep {best_eval['epoch']:.0f}")
    ax.legend()
ax.set_xlabel("epoch"); ax.set_ylabel("F1 macro"); ax.set_title("Dev F1 (macro)")
ax.grid(alpha=0.3)

# Per-class F1 evolution.
ax = axes[1, 0]
for col, color, lbl in [
    ("eval_f1_clean",     "#4CAF50", "CLEAN"),
    ("eval_f1_offensive", "#FF9800", "OFFENSIVE"),
    ("eval_f1_hate",      "#F44336", "HATE"),
]:
    if col in eval_df_log.columns:
        ax.plot(eval_df_log["epoch"], eval_df_log[col], "o-", color=color, label=lbl)
ax.set_xlabel("epoch"); ax.set_ylabel("F1"); ax.set_title("Per-class dev F1")
ax.legend(); ax.grid(alpha=0.3)

# Learning rate schedule.
ax = axes[1, 1]
if not train_df_log.empty and "learning_rate" in train_df_log.columns:
    ax.plot(train_df_log["step"], train_df_log["learning_rate"], "-", color="#9D7BBA")
    ax.set_xlabel("step"); ax.set_ylabel("learning rate"); ax.set_title("LR schedule (linear w/ warmup)")
    ax.grid(alpha=0.3)

fig.suptitle(f"PhoBERT fine-tuning — best dev F1_macro = {phobert_metrics['f1_macro']:.4f}",
             y=1.02, fontsize=13)
fig_path = FIG_DIR / "19_phobert_training_curves.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"✓ saved → {fig_path}")


## E. 3-way comparison (LR vs BiLSTM v2 vs PhoBERT)

Headline numbers, per-class breakdown, confusion-matrix triptych, and a
training-time vs F1 trade-off plot. Then write `report/week4_phase2_summary.md`.

In [ ]:
# Load LR champion + BiLSTM v2 stats (with sensible fallbacks).
LR_DEFAULT = {
    "model": "LR + char-ngram (Week 3)",
    "f1_macro":      0.6183,
    "f1_clean":      0.9144,
    "f1_offensive":  0.3976,
    "f1_hate":       0.5428,
    "train_time":    2.0,
    "inference_time": 0.0,
    "params_trainable": 25_000,
}
BILSTM_V2_DEFAULT = {
    "model": "BiLSTM v2 (Exp3 strong_reg)",
    "f1_macro":      0.6195,
    "f1_clean":      0.8981,
    "f1_offensive":  0.4205,
    "f1_hate":       0.5399,
    "train_time":    0.0,
    "inference_time": 0.0,
    "params_trainable": 3_500_000,
}

# LR — try week3 results JSON.
lr_row = LR_DEFAULT.copy()
wk3 = RESULTS_DIR / "week3_final_metrics.json"
if wk3.exists():
    try:
        with open(wk3, "r", encoding="utf-8") as f:
            j = json.load(f)
        cands = []
        if isinstance(j, dict):
            for k, v in j.items():
                if isinstance(v, dict) and "f1_macro" in v:
                    cands.append((k, v))
        best = max(cands, key=lambda kv: (
            ("char" in kv[0].lower() or "char" in str(kv[1].get("model","")).lower())
            + ("lr"   in kv[0].lower() or "logreg" in kv[0].lower()),
            kv[1].get("f1_macro", 0),
        ), default=None)
        if best:
            name, m = best
            lr_row.update({
                "model":        m.get("model", name),
                "f1_macro":     float(m.get("f1_macro", lr_row["f1_macro"])),
                "f1_clean":     float(m.get("f1_clean", lr_row["f1_clean"])),
                "f1_offensive": float(m.get("f1_offensive", lr_row["f1_offensive"])),
                "f1_hate":      float(m.get("f1_hate", lr_row["f1_hate"])),
                "train_time":   float(m.get("train_time", lr_row["train_time"])),
                "inference_time": float(m.get("inference_time", lr_row["inference_time"])),
            })
            print(f"LR row loaded from {wk3.name} ('{name}').")
    except Exception as e:
        print(f"(could not parse {wk3.name}: {e}; using LR defaults)")

# BiLSTM v2 — try the tuning checkpoint.
bilstm_v2_row = BILSTM_V2_DEFAULT.copy()
v2_ckpt = DL_DIR / "bilstm_v2_best.pt"
if v2_ckpt.exists():
    try:
        ck = torch.load(v2_ckpt, weights_only=False, map_location="cpu")
        m = ck.get("metrics", {})
        bilstm_v2_row.update({
            "f1_macro":         float(m.get("f1_macro", bilstm_v2_row["f1_macro"])),
            "f1_clean":         float(m.get("f1_clean", bilstm_v2_row["f1_clean"])),
            "f1_offensive":     float(m.get("f1_offensive", bilstm_v2_row["f1_offensive"])),
            "f1_hate":          float(m.get("f1_hate", bilstm_v2_row["f1_hate"])),
            "params_trainable": int(ck.get("param_info", {}).get("trainable", bilstm_v2_row["params_trainable"])),
        })
        print(f"BiLSTM v2 row loaded from {v2_ckpt.name} (exp={ck.get('experiment','?')}, ep={ck.get('best_epoch','?')}).")
    except Exception as e:
        print(f"(could not parse {v2_ckpt.name}: {e}; using BiLSTM defaults)")

# PhoBERT row from this notebook.
phobert_row = {
    "model": phobert_metrics["model"],
    "f1_macro":         float(phobert_metrics["f1_macro"]),
    "f1_clean":         float(phobert_metrics["f1_clean"]),
    "f1_offensive":     float(phobert_metrics["f1_offensive"]),
    "f1_hate":          float(phobert_metrics["f1_hate"]),
    "train_time":       float(phobert_metrics["train_time"]),
    "inference_time":   float(phobert_metrics["inference_time"]),
    "params_trainable": int(trainable_params),
}

rows = [lr_row, bilstm_v2_row, phobert_row]

def _row_view(r):
    return {
        "model":         r["model"],
        "f1_macro":      r["f1_macro"],
        "f1_CLEAN":      r["f1_clean"],
        "f1_OFF":        r["f1_offensive"],
        "f1_HATE":       r["f1_hate"],
        "train_s":       r["train_time"],
        "infer_s":       r.get("inference_time", 0.0),
        "params":        r["params_trainable"],
    }

compare_df = pd.DataFrame([_row_view(r) for r in rows])
print("\n3-way comparison (DEV):\n")
print(compare_df.to_string(
    index=False,
    formatters={
        "f1_macro":  "{:.4f}".format,
        "f1_CLEAN":  "{:.4f}".format,
        "f1_OFF":    "{:.4f}".format,
        "f1_HATE":   "{:.4f}".format,
        "train_s":   "{:.1f}".format,
        "infer_s":   "{:.2f}".format,
        "params":    "{:,.0f}".format,
    }))

# Deltas vs LR.
d_lr_macro = phobert_row["f1_macro"] - lr_row["f1_macro"]
d_lr_off   = phobert_row["f1_offensive"] - lr_row["f1_offensive"]
d_lr_hate  = phobert_row["f1_hate"] - lr_row["f1_hate"]
# vs BiLSTM v2.
d_bl_macro = phobert_row["f1_macro"] - bilstm_v2_row["f1_macro"]
d_bl_off   = phobert_row["f1_offensive"] - bilstm_v2_row["f1_offensive"]
d_bl_hate  = phobert_row["f1_hate"] - bilstm_v2_row["f1_hate"]

print(f"\nΔ vs LR champion:")
print(f"  f1_macro: {d_lr_macro:+.4f}  ({d_lr_macro/lr_row['f1_macro']*100:+.1f}%)")
print(f"  f1_OFF  : {d_lr_off:+.4f}")
print(f"  f1_HATE : {d_lr_hate:+.4f}")
print(f"\nΔ vs BiLSTM v2:")
print(f"  f1_macro: {d_bl_macro:+.4f}  ({d_bl_macro/bilstm_v2_row['f1_macro']*100:+.1f}%)")
print(f"  f1_OFF  : {d_bl_off:+.4f}")
print(f"  f1_HATE : {d_bl_hate:+.4f}")


In [ ]:
# 2x2 grid: macro bar, per-class grouped bar, trade-off, confusion side-by-side
fig = plt.figure(figsize=(14, 11), constrained_layout=True)
gs  = fig.add_gridspec(3, 3)

names    = [r["model"] for r in rows]
labels   = ["CLEAN", "OFFENSIVE", "HATE"]
palette  = ["#9D7BBA", "#4C78A8", "#54A24B"]   # LR / BiLSTM / PhoBERT

# (top-left) F1 macro bar
ax1 = fig.add_subplot(gs[0, 0])
f1s = [r["f1_macro"] for r in rows]
bars = ax1.bar(range(len(rows)), f1s, color=palette)
ax1.set_xticks(range(len(rows)))
ax1.set_xticklabels(["LR", "BiLSTM v2", "PhoBERT"], fontsize=10)
ax1.set_ylim(0.5, max(0.75, max(f1s) + 0.05))
ax1.set_ylabel("Dev F1 macro"); ax1.set_title("F1 macro (DEV)")
for bar, val in zip(bars, f1s):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.005, f"{val:.4f}",
             ha="center", va="bottom", fontsize=9)
ax1.grid(alpha=0.3, axis="y")

# (top-middle) per-class grouped bar
ax2 = fig.add_subplot(gs[0, 1:])
x = np.arange(len(labels))
w = 0.27
for i, r in enumerate(rows):
    vals = [r["f1_clean"], r["f1_offensive"], r["f1_hate"]]
    ax2.bar(x + (i - 1) * w, vals, w, label=["LR", "BiLSTM v2", "PhoBERT"][i], color=palette[i])
ax2.set_xticks(x); ax2.set_xticklabels(labels)
ax2.set_ylabel("F1"); ax2.set_title("Per-class F1 (DEV)")
ax2.legend(loc="lower right"); ax2.grid(alpha=0.3, axis="y")

# (middle) trade-off: train time (log) vs f1_macro
ax3 = fig.add_subplot(gs[1, 0])
xs = [max(r["train_time"], 0.1) for r in rows]   # avoid log(0)
ys = [r["f1_macro"]            for r in rows]
ax3.scatter(xs, ys, c=palette, s=160, edgecolor="white", linewidth=1.5, zorder=3)
for x_, y_, n in zip(xs, ys, ["LR", "BiLSTM v2", "PhoBERT"]):
    ax3.annotate(n, (x_, y_), xytext=(7, 7), textcoords="offset points", fontsize=9)
ax3.set_xscale("log")
ax3.set_xlabel("Train time (s, log)")
ax3.set_ylabel("Dev F1 macro")
ax3.set_title("Train-time vs F1 trade-off")
ax3.grid(alpha=0.3, which="both")

# (middle/right) param count (log) vs f1_macro
ax4 = fig.add_subplot(gs[1, 1])
xs2 = [max(r["params_trainable"], 1) for r in rows]
ax4.scatter(xs2, ys, c=palette, s=160, edgecolor="white", linewidth=1.5, zorder=3)
for x_, y_, n in zip(xs2, ys, ["LR", "BiLSTM v2", "PhoBERT"]):
    ax4.annotate(n, (x_, y_), xytext=(7, 7), textcoords="offset points", fontsize=9)
ax4.set_xscale("log")
ax4.set_xlabel("Trainable params (log)")
ax4.set_ylabel("Dev F1 macro")
ax4.set_title("Params vs F1 trade-off")
ax4.grid(alpha=0.3, which="both")

# (middle/right2) summary text
ax5 = fig.add_subplot(gs[1, 2]); ax5.axis("off")
ax5.text(0.0, 1.0, "Headline deltas:", fontsize=11, weight="bold", va="top")
text = (
    f"PhoBERT vs LR:\n"
    f"  ΔF1_macro = {d_lr_macro:+.4f}\n"
    f"  ΔF1_OFF   = {d_lr_off:+.4f}\n"
    f"  ΔF1_HATE  = {d_lr_hate:+.4f}\n\n"
    f"PhoBERT vs BiLSTM v2:\n"
    f"  ΔF1_macro = {d_bl_macro:+.4f}\n"
    f"  ΔF1_OFF   = {d_bl_off:+.4f}\n"
    f"  ΔF1_HATE  = {d_bl_hate:+.4f}\n\n"
    f"Train time ratio:\n"
    f"  PhoBERT / LR     = {phobert_row['train_time'] / max(lr_row['train_time'], 0.1):.0f}x\n"
    f"  PhoBERT / BiLSTM = {phobert_row['train_time'] / max(bilstm_v2_row['train_time'], 0.1):.1f}x"
)
ax5.text(0.0, 0.92, text, fontsize=10, family="monospace", va="top")

# (bottom row) three confusion matrices side by side
from sklearn.metrics import confusion_matrix
for j, (name, color) in enumerate(zip(["LR (Week 3)", "BiLSTM v2", "PhoBERT"], palette)):
    ax = fig.add_subplot(gs[2, j])
    if j == 2:
        cm = confusion_matrix(y_dev_true, y_dev_pred, labels=[0, 1, 2])
    else:
        # We don't have raw preds for LR/BiLSTM saved here; show counts only via per-class F1.
        ax.axis("off")
        ax.set_title(f"{name}\nF1m={[lr_row, bilstm_v2_row][j]['f1_macro']:.4f}\n"
                     f"(per-sample preds not saved in 04c)", fontsize=10)
        continue
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks([0,1,2]); ax.set_xticklabels(labels, fontsize=8)
    ax.set_yticks([0,1,2]); ax.set_yticklabels(labels, fontsize=8)
    ax.set_title(f"{name}\nF1m={phobert_row['f1_macro']:.4f}", fontsize=10)
    for i in range(3):
        for jj in range(3):
            ax.text(jj, i, f"{cm_norm[i,jj]:.1%}\n({cm[i,jj]})",
                    ha="center", va="center", fontsize=8,
                    color="white" if cm_norm[i,jj] > 0.5 else "black")
    ax.set_xlabel("predicted")
    if j == 0:
        ax.set_ylabel("true")

fig.suptitle("Week 4 — DEV comparison: LR vs BiLSTM v2 vs PhoBERT-base-v2",
             fontsize=13, y=1.01)
fig_path = FIG_DIR / "20_three_way_comparison.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"✓ saved → {fig_path}")


In [ ]:
# Headline + cost + failure-modes + bridge to Week 5.
champion = max(rows, key=lambda r: r["f1_macro"])
champion_name = champion["model"]

# Failure mode: most common PhoBERT confusion class.
from sklearn.metrics import confusion_matrix as _cm
cm_ph = _cm(y_dev_true, y_dev_pred, labels=[0, 1, 2])
# False-negatives per class on the diagonal: cm_ph[i, :].sum() - cm_ph[i, i] = FN for class i.
fn = {i: int(cm_ph[i, :].sum() - cm_ph[i, i]) for i in range(3)}
fp = {i: int(cm_ph[:, i].sum() - cm_ph[i, i]) for i in range(3)}
per_class = [("CLEAN", phobert_row["f1_clean"]),
             ("OFFENSIVE", phobert_row["f1_offensive"]),
             ("HATE", phobert_row["f1_hate"])]
weakest_name, weakest_f1 = min(per_class, key=lambda kv: kv[1])

off_to_clean = int(cm_ph[1, 0])  # OFFENSIVE classified as CLEAN
off_total    = int(cm_ph[1, :].sum())

lines = []
lines.append("# Week 4 — Phase 2 summary: PhoBERT-base-v2 fine-tuning")
lines.append("")
lines.append("_Auto-generated by `notebooks/04c_phobert.ipynb` on the best epoch by dev `f1_macro`._")
lines.append("")
lines.append("## 1. Headline results (DEV)")
lines.append("")
lines.append("| Model | F1 macro | F1 CLEAN | F1 OFF | F1 HATE | Train time | Inference | Params |")
lines.append("|---|---|---|---|---|---|---|---|")
for r in rows:
    lines.append(
        f"| {r['model']} "
        f"| {r['f1_macro']:.4f} "
        f"| {r['f1_clean']:.4f} "
        f"| {r['f1_offensive']:.4f} "
        f"| {r['f1_hate']:.4f} "
        f"| {r['train_time']:.0f}s "
        f"| {r.get('inference_time', 0.0):.2f}s "
        f"| {r['params_trainable']:,} |"
    )
lines.append("")

# Section 2 — PhoBERT improvement
lines.append("## 2. PhoBERT improvement")
lines.append("")
lines.append(f"- vs LR champion : Δ f1_macro = {d_lr_macro:+.4f} "
             f"({d_lr_macro/lr_row['f1_macro']*100:+.1f}%) — OFF {d_lr_off:+.4f}, HATE {d_lr_hate:+.4f}")
lines.append(f"- vs BiLSTM v2   : Δ f1_macro = {d_bl_macro:+.4f} "
             f"({d_bl_macro/bilstm_v2_row['f1_macro']*100:+.1f}%) — OFF {d_bl_off:+.4f}, HATE {d_bl_hate:+.4f}")
lines.append("")
if d_lr_macro > 0:
    lines.append(f"- ✓ PhoBERT beats LR by {d_lr_macro:+.4f} on dev `f1_macro`.")
if d_bl_macro > 0:
    lines.append(f"- ✓ PhoBERT beats BiLSTM v2 by {d_bl_macro:+.4f} on dev `f1_macro`.")
if d_bl_macro < 0:
    lines.append(f"- ⚠ PhoBERT trails BiLSTM v2 by {d_bl_macro:+.4f} — unexpected; check fp16 underflow / lr.")
lines.append("")

# Section 3 — Cost
lines.append("## 3. Cost analysis")
lines.append("")
lines.append(f"- Train time: PhoBERT {phobert_row['train_time']:.0f}s "
             f"vs LR {lr_row['train_time']:.0f}s "
             f"= **{phobert_row['train_time']/max(lr_row['train_time'],0.1):.0f}× slower**.")
lines.append(f"- Inference: PhoBERT {phobert_row['inference_time']:.2f}s for {len(y_dev_true):,} dev samples "
             f"({phobert_row['inference_time']/len(y_dev_true)*1000:.2f} ms/sample on GPU fp16).")
lines.append(f"- Model size: PhoBERT {phobert_row['params_trainable']:,} params "
             f"vs LR ~{lr_row['params_trainable']:,} params "
             f"= **{phobert_row['params_trainable']/max(lr_row['params_trainable'],1):,.0f}× larger**.")
lines.append("")
lines.append("- **Worth the cost?** Depends on deployment: for an offline moderation queue where "
             f"~{phobert_row['inference_time']/len(y_dev_true)*1000:.1f} ms/sample is acceptable and the f1_macro "
             f"gain ({d_lr_macro:+.4f}) matters, yes. For a real-time chat filter at thousands of QPS, "
             "LR + char-ngram remains attractive — keep PhoBERT as a fallback for borderline cases.")
lines.append("")

# Section 4 — failure modes
lines.append("## 4. PhoBERT failure modes")
lines.append("")
lines.append(f"- Weakest class: **{weakest_name}** with F1 = {weakest_f1:.4f}.")
lines.append(f"- OFFENSIVE → CLEAN false negatives: {off_to_clean} / {off_total} OFFENSIVE samples "
             f"({100*off_to_clean/max(off_total,1):.1f}%).")
lines.append(f"- Confusion matrix (DEV, raw counts):")
lines.append("")
lines.append("|  | pred CLEAN | pred OFF | pred HATE |")
lines.append("|---|---|---|---|")
for i, name in enumerate(["true CLEAN", "true OFF", "true HATE"]):
    lines.append(f"| {name} | {int(cm_ph[i,0])} | {int(cm_ph[i,1])} | {int(cm_ph[i,2])} |")
lines.append("")
lines.append("- Cases where LR was right but PhoBERT was wrong are saved in "
             "`results/phobert_dev_predictions.csv` for Week-5 error analysis "
             "(join with the Week-3 LR predictions to surface the disagreement set).")
lines.append("")

# Section 5 — Bridge to Week 5
lines.append("## 5. Bridge to Week 5")
lines.append("")
lines.append(f"- **Champion (on DEV)**: `{champion_name}` (f1_macro = {champion['f1_macro']:.4f}).")
lines.append(f"- For TEST evaluation (Week-5 Prompt 4): use `models/dl/phobert_best/` as the "
             f"frozen artefact — checkpoint inside is the epoch that maximised dev f1_macro.")
lines.append(f"- Predictions for error analysis: `results/phobert_dev_predictions.csv`.")
lines.append(f"- Hypotheses for Week-5 error analysis:")
lines.append(f"  * Sarcasm / negation handling (PhoBERT should excel; check vs LR)")
lines.append(f"  * Code-switched English-Vietnamese slurs (likely both models struggle)")
lines.append(f"  * Very short toxic samples (length<3 — filtered out of train; check recall)")
lines.append(f"  * Threshold tuning per class — current logits use argmax; tuned thresholds on "
             f"OFFENSIVE/HATE may shift the F1 budget.")
lines.append("")

# Section 6 — Artefacts
lines.append("## 6. Artefacts")
lines.append("")
lines.append(f"- `models/dl/phobert_best/`                 — final model + tokenizer (use this for TEST)")
lines.append(f"- `results/phobert_training_log.json`")
lines.append(f"- `results/phobert_dev_predictions.csv`")
lines.append(f"- `results/figures/18_phobert_dev_cm.png`")
lines.append(f"- `results/figures/19_phobert_training_curves.png`")
lines.append(f"- `results/figures/20_three_way_comparison.png`")

SUMMARY = REPORT_DIR / "week4_phase2_summary.md"
SUMMARY.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"✓ saved → {SUMMARY}")
print()
print(SUMMARY.read_text(encoding="utf-8"))
